# MultiBiz Forecasting Models - Google Colab Training
## XGBoost & Prophet for Supply Consumption Forecasting

This notebook trains demand forecasting models for the Multibliz POS System.

**Models Trained:**
- XGBoost Regressor (Gradient Boosting)
- Facebook Prophet (Time-Series)

**Duration:** ~5-10 minutes
**GPU Required:** No (CPU-based)
**Storage:** 2-3 GB (includes models + visualizations)

## Step 1: Install Required Libraries

In [ ]:
# Install required packages
!pip install -q pandas numpy scikit-learn xgboost prophet joblib seaborn matplotlib

print("✓ All libraries installed successfully!")

## Step 2: Mount Google Drive & Load Data

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Navigate to your project folder
# ⚠️ CHANGE THIS PATH to match your Google Drive folder structure
project_path = '/content/drive/MyDrive/Multibliz POS System'

# Check if folder exists
if os.path.exists(project_path):
    os.chdir(project_path)
    print(f"✓ Working directory: {os.getcwd()}")
    print(f"✓ Files available: {len(os.listdir())} items")
else:
    print("⚠️ Folder not found. Update the project_path variable above.")
    print("📁 Your Drive folders:")
    print(os.listdir('/content/drive/MyDrive'))

## Step 3: Import Libraries & Configure

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
import xgboost as xgb
from prophet import Prophet
import joblib
import warnings

warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

print("✓ All imports successful!")
print(f"✓ XGBoost version: {xgb.__version__}")
print(f"✓ Prophet version: {Prophet.__module__}")

## Step 4: Load & Preprocess Data

⚠️ **Note:** Update the file path to match your dataset location

In [ ]:
# Option 1: Load from CSV file (if you have the Filtered_Printing_Business.csv)
try:
    df = pd.read_csv('data/datasets/Filtered_Printing_Business.csv', encoding='latin-1')
    print(f"✓ Loaded from CSV: {len(df)} records")
except FileNotFoundError:
    # Option 2: Generate sample data if file doesn't exist
    print("⚠️ CSV file not found. Generating synthetic data...")
    
    np.random.seed(42)
    dates = pd.date_range(start='2023-01-01', end='2024-12-31', freq='D')
    
    # Create synthetic sales data
    data = []
    for date in dates:
        # More sales on weekdays, less on weekends
        day_type = date.dayofweek
        base_quantity = 15 if day_type < 5 else 8
        
        # Add some randomness and seasonal patterns
        quantity = base_quantity + np.random.normal(0, 3) + np.sin(date.dayofyear / 365 * 2 * np.pi) * 5
        quantity = max(0, int(quantity))
        
        if quantity > 0:  # Only include days with sales
            data.append({
                'Order Date': date,
                'Quantity': quantity,
                'Sales': quantity * np.random.uniform(100, 500),
                'Category': np.random.choice(['Paper', 'Binders', 'Labels', 'Art', 'Fasteners']),
                'Product Name': f'Product_{np.random.randint(1, 50)}'
            })
    
    df = pd.DataFrame(data)
    print(f"✓ Generated synthetic data: {len(df)} records")

# Display data info
print(f"\n📊 Dataset Information:")
print(f"   Date range: {df['Order Date'].min()} to {df['Order Date'].max()}")
print(f"   Categories: {df['Category'].unique() if 'Category' in df.columns else 'N/A'}")
print(f"   Total records: {len(df):,}")
print(f"\n{df.head()}")

## Step 5: Data Aggregation - Daily Consumption

In [ ]:
# Convert Order Date to datetime
df['Order Date'] = pd.to_datetime(df['Order Date'])

# Aggregate daily consumption
daily_sales = df.groupby('Order Date').agg({
    'Quantity': 'sum',
    'Sales': 'sum'
}).reset_index()

daily_sales.columns = ['Date', 'Quantity_Sold', 'Sales_Value']

# Fill missing dates with 0 (no sales that day)
date_range = pd.date_range(start=daily_sales['Date'].min(), 
                           end=daily_sales['Date'].max(), 
                           freq='D')
daily_sales = daily_sales.set_index('Date').reindex(date_range, fill_value=0).reset_index()
daily_sales.columns = ['Date', 'Quantity_Sold', 'Sales_Value']

print(f"📈 Daily Consumption Statistics:")
print(f"   Total days: {len(daily_sales)}")
print(f"   Average daily: {daily_sales['Quantity_Sold'].mean():.2f} units")
print(f"   Maximum daily: {daily_sales['Quantity_Sold'].max():.0f} units")
print(f"   Minimum daily: {daily_sales['Quantity_Sold'].min():.0f} units")
print(f"   Days with zero sales: {(daily_sales['Quantity_Sold'] == 0).sum()}")

# Visualize daily consumption
plt.figure(figsize=(14, 5))
plt.plot(daily_sales['Date'], daily_sales['Quantity_Sold'], linewidth=1.5, color='#2E86AB')
plt.title('Daily Consumption Over Time', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontweight='bold')
plt.ylabel('Quantity Consumed (Units)', fontweight='bold')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"\n✓ Data aggregated and visualized")

## Step 6: Feature Engineering for XGBoost

In [ ]:
# Create features for XGBoost
xgb_data = daily_sales.copy()

# ⏰ Temporal Features
xgb_data['Day_of_Week'] = xgb_data['Date'].dt.dayofweek        # 0=Monday, 6=Sunday
xgb_data['Month'] = xgb_data['Date'].dt.month                  # 1-12
xgb_data['Day_of_Year'] = xgb_data['Date'].dt.dayofyear        # 1-365
xgb_data['Quarter'] = xgb_data['Date'].dt.quarter              # 1-4
xgb_data['Week_of_Year'] = xgb_data['Date'].dt.isocalendar().week  # 1-52

# 📊 Lag Features (previous days' sales)
for lag in [1, 7, 14, 30]:
    xgb_data[f'Lag_{lag}'] = xgb_data['Quantity_Sold'].shift(lag)

# 📈 Rolling Statistics
xgb_data['Rolling_Mean_7'] = xgb_data['Quantity_Sold'].rolling(window=7, min_periods=1).mean()
xgb_data['Rolling_Mean_30'] = xgb_data['Quantity_Sold'].rolling(window=30, min_periods=1).mean()
xgb_data['Rolling_Std_7'] = xgb_data['Quantity_Sold'].rolling(window=7, min_periods=1).std()

# Remove rows with NaN from lag features
xgb_data = xgb_data.dropna()

print("✓ Features Created:")
print(f"   Temporal: Day_of_Week, Month, Day_of_Year, Quarter, Week_of_Year")
print(f"   Lag features: 1, 7, 14, 30 days")
print(f"   Rolling statistics: 7-day and 30-day moving averages/std")
print(f"\n   Final dataset size: {len(xgb_data)} days")
print(f"\n{xgb_data.head()}")

## Step 7: Train Prophet Model

In [ ]:
# Prepare data for Prophet (needs 'ds' and 'y' columns)
prophet_data = daily_sales[['Date', 'Quantity_Sold']].copy()
prophet_data.columns = ['ds', 'y']

print("📅 Training Prophet Model...")
print("   This may take 1-2 minutes...\n")

# Create and train Prophet model
prophet_model = Prophet(
    yearly_seasonality=True,           # Capture annual patterns
    weekly_seasonality=True,           # Capture weekly patterns
    daily_seasonality=False,           # Not needed for daily data
    changepoint_prior_scale=0.05,      # Trend flexibility
    seasonality_prior_scale=10.0,      # Seasonality strength
    interval_width=0.95                # 95% confidence intervals
)

# Add custom seasonality
prophet_model.add_seasonality(name='quarterly', period=91.25, fourier_order=5)

# Train
with suppress_stdout_stderr():
    prophet_model.fit(prophet_data)

print("✓ Prophet model trained successfully!")

# Make predictions on historical data
future_dates = prophet_model.make_future_dataframe(periods=0)
prophet_forecast = prophet_model.predict(future_dates)

print(f"\n📊 Prophet Forecast Sample:")
print(prophet_forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail())

### Helper function for suppressing Prophet output

In [ ]:
import sys
from contextlib import contextmanager
import os

@contextmanager
def suppress_stdout_stderr():
    """Suppress stdout and stderr output"""
    save_stdout = sys.stdout
    save_stderr = sys.stderr
    sys.stdout = open(os.devnull, 'w')
    sys.stderr = open(os.devnull, 'w')
    try:
        yield
    finally:
        sys.stdout = save_stdout
        sys.stderr = save_stderr

## Step 8: Train XGBoost Model

In [ ]:
# Define features and target
feature_columns = ['Day_of_Week', 'Month', 'Day_of_Year', 'Quarter', 'Week_of_Year',
                   'Lag_1', 'Lag_7', 'Lag_14', 'Lag_30',
                   'Rolling_Mean_7', 'Rolling_Mean_30', 'Rolling_Std_7']

X = xgb_data[feature_columns]
y = xgb_data['Quantity_Sold']

# Split data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False  # Don't shuffle time-series!
)

print("🎯 Training XGBoost Model...")
print(f"   Training set: {len(X_train)} samples")
print(f"   Test set: {len(X_test)} samples\n")

# Create and train XGBoost model
xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=200,              # Number of boosting rounds
    learning_rate=0.05,            # Learning rate
    max_depth=6,                   # Tree depth
    min_child_weight=3,            # Min samples per leaf
    subsample=0.8,                 # Subsample fraction
    colsample_bytree=0.8,          # Feature sample fraction
    random_state=42,
    n_jobs=-1,                     # Use all CPU cores
    verbosity=0
)

# Train the model
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

print("✓ XGBoost model trained successfully!")

## Step 9: Model Evaluation & Metrics

In [ ]:
print("\n" + "="*80)
print("MODEL EVALUATION METRICS")
print("="*80)

# ===== PROPHET EVALUATION =====
prophet_predictions = prophet_forecast[['ds', 'yhat']].merge(
    prophet_data, on='ds', how='inner'
)

prophet_mae = mean_absolute_error(prophet_predictions['y'], prophet_predictions['yhat'])
prophet_rmse = np.sqrt(mean_squared_error(prophet_predictions['y'], prophet_predictions['yhat']))

print("\n📅 PROPHET MODEL:")
print(f"   MAE (Mean Absolute Error):  {prophet_mae:.2f} units")
print(f"   RMSE (Root Mean Squared Error): {prophet_rmse:.2f} units")
print(f"   → Predictions deviate by ±{prophet_mae:.2f} units on average")

# ===== XGBOOST EVALUATION =====
y_pred_train = xgb_model.predict(X_train)
y_pred_test = xgb_model.predict(X_test)

xgb_mae_train = mean_absolute_error(y_train, y_pred_train)
xgb_rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
xgb_mae_test = mean_absolute_error(y_test, y_pred_test)
xgb_rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))

print("\n🚀 XGBOOST MODEL:")
print("   Training Set:")
print(f"     MAE:  {xgb_mae_train:.2f} units")
print(f"     RMSE: {xgb_rmse_train:.2f} units")
print("   Test Set:")
print(f"     MAE:  {xgb_mae_test:.2f} units")
print(f"     RMSE: {xgb_rmse_test:.2f} units")
print(f"   → Test predictions deviate by ±{xgb_mae_test:.2f} units on average")

# Check for overfitting
print("\n🔍 OVERFITTING CHECK:")
if xgb_mae_test > xgb_mae_train * 1.5:
    print("   ⚠️  WARNING: Possible overfitting detected (test error >> training error)")
else:
    print("   ✓ Model generalizes well (no significant overfitting)")

print("\n" + "="*80)

## Step 10: Feature Importance Analysis

In [ ]:
# Get feature importance
feature_importance = pd.DataFrame({
    'Feature': feature_columns,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\n🎯 XGBOOST FEATURE IMPORTANCE (Top 10)\n")
print(feature_importance.head(10).to_string(index=False))
print("\n→ These features have the strongest impact on predictions")

# Visualize feature importance
plt.figure(figsize=(10, 6))
top_features = feature_importance.head(10)
plt.barh(top_features['Feature'], top_features['Importance'], color='#F18F01')
plt.xlabel('Importance Score', fontweight='bold')
plt.ylabel('Feature', fontweight='bold')
plt.title('XGBoost: Top 10 Most Important Features', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## Step 11: Predictions vs Actual - Visualization

In [ ]:
# Get dates for test set
test_dates = xgb_data.iloc[-len(y_test):]['Date'].values

# Create comparison plot
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# ===== XGBOOST PLOT =====
axes[0].plot(test_dates, y_test.values, label='Actual Consumption',
            linewidth=2, color='#2E86AB', marker='o', markersize=4)
axes[0].plot(test_dates, y_pred_test, label='XGBoost Prediction',
            linewidth=2, color='#A23B72', linestyle='--', marker='s', markersize=4)
axes[0].set_title(f'XGBoost: Predicted vs Actual (Test Set)\nMAE: {xgb_mae_test:.2f} | RMSE: {xgb_rmse_test:.2f}',
                 fontsize=12, fontweight='bold')
axes[0].set_xlabel('Date', fontweight='bold')
axes[0].set_ylabel('Quantity (Units)', fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].tick_params(axis='x', rotation=45)

# ===== PROPHET PLOT =====
prophet_test = prophet_predictions.tail(len(y_test))
axes[1].plot(prophet_test['ds'], prophet_test['y'], label='Actual Consumption',
            linewidth=2, color='#2E86AB', marker='o', markersize=4)
axes[1].plot(prophet_test['ds'], prophet_test['yhat'], label='Prophet Prediction',
            linewidth=2, color='#06A77D', linestyle='--', marker='^', markersize=4)
axes[1].fill_between(prophet_test['ds'],
                     prophet_test['yhat_lower'],
                     prophet_test['yhat_upper'],
                     alpha=0.2, color='#06A77D', label='95% Confidence Interval')
axes[1].set_title(f'Prophet: Predicted vs Actual (Test Set)\nMAE: {prophet_mae:.2f} | RMSE: {prophet_rmse:.2f}',
                 fontsize=12, fontweight='bold')
axes[1].set_xlabel('Date', fontweight='bold')
axes[1].set_ylabel('Quantity (Units)', fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("✓ Prediction visualizations generated")

## Step 12: Prophet Components Visualization

In [ ]:
# Show Prophet components (trend + seasonality)
print("📊 Prophet Forecast Components Analysis\n")

fig = prophet_model.plot_components(prophet_forecast, figsize=(14, 10))
fig.suptitle('Prophet Model: Trend & Seasonality Components', 
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print("✓ Prophet component visualization completed")

## Step 13: Future Forecasts (Next 30 Days)

In [ ]:
print("\n" + "="*80)
print("FUTURE FORECASTS (Next 30 Days)")
print("="*80)

# Get last date
last_date = daily_sales['Date'].max()
forecast_dates = pd.date_range(start=last_date + timedelta(days=1), periods=30, freq='D')

# ===== XGBOOST FUTURE FORECAST =====
xgb_future_preds = []
for forecast_date in forecast_dates:
    day_of_week = forecast_date.dayofweek
    month = forecast_date.month
    day_of_year = forecast_date.dayofyear
    quarter = forecast_date.quarter
    week_of_year = forecast_date.isocalendar()[1]
    
    # Use recent lags
    lag_1 = y_test.iloc[-1] if len(y_test) > 0 else y.mean()
    lag_7 = y.iloc[-7] if len(y) > 6 else y.mean()
    lag_14 = y.iloc[-14] if len(y) > 13 else y.mean()
    lag_30 = y.iloc[-30] if len(y) > 29 else y.mean()
    
    rolling_mean_7 = y.tail(7).mean()
    rolling_mean_30 = y.tail(30).mean()
    rolling_std_7 = y.tail(7).std()
    
    features = pd.DataFrame([{
        'Day_of_Week': day_of_week,
        'Month': month,
        'Day_of_Year': day_of_year,
        'Quarter': quarter,
        'Week_of_Year': week_of_year,
        'Lag_1': lag_1,
        'Lag_7': lag_7,
        'Lag_14': lag_14,
        'Lag_30': lag_30,
        'Rolling_Mean_7': rolling_mean_7,
        'Rolling_Mean_30': rolling_mean_30,
        'Rolling_Std_7': rolling_std_7
    }])
    
    pred = max(0, int(xgb_model.predict(features)[0]))
    xgb_future_preds.append(pred)

# ===== PROPHET FUTURE FORECAST =====
future = prophet_model.make_future_dataframe(periods=30)
with suppress_stdout_stderr():
    prophet_future_forecast = prophet_model.predict(future)

prophet_future = prophet_future_forecast[prophet_future_forecast['ds'] > last_date]
prophet_future_preds = [max(0, int(val)) for val in prophet_future['yhat'].values]

# Create summary table
forecast_summary = pd.DataFrame({
    'Date': forecast_dates,
    'XGBoost': xgb_future_preds,
    'Prophet': prophet_future_preds,
    'Average': [int((x + y) / 2) for x, y in zip(xgb_future_preds, prophet_future_preds)]
})

print("\n📈 Next 30 Days Forecast:")
print(forecast_summary.head(15).to_string(index=False))
print("\n... (showing first 15 days)\n")

print(f"Summary Statistics (Next 30 Days):")
print(f"   XGBoost Average:  {np.mean(xgb_future_preds):.0f} units/day")
print(f"   Prophet Average:  {np.mean(prophet_future_preds):.0f} units/day")
print(f"   Ensemble Average: {forecast_summary['Average'].mean():.0f} units/day")

## Step 14: Visualize Future Forecasts

In [ ]:
# Plot recent + future predictions
plt.figure(figsize=(14, 6))

# Recent historical data
recent_days = 60
recent_data = daily_sales.tail(recent_days)

# Plot historical
plt.plot(recent_data['Date'], recent_data['Quantity_Sold'], 
        label='Historical Data', linewidth=2, color='#2E86AB', marker='o', markersize=4)

# Plot future forecasts
plt.plot(forecast_dates, xgb_future_preds, 
        label='XGBoost Forecast', linewidth=2, color='#A23B72', 
        linestyle='--', marker='s', markersize=4)
plt.plot(forecast_dates, prophet_future_preds, 
        label='Prophet Forecast', linewidth=2, color='#06A77D', 
        linestyle='-.', marker='^', markersize=4)
plt.plot(forecast_dates, forecast_summary['Average'], 
        label='Ensemble (Average)', linewidth=2.5, color='#F18F01', 
        linestyle=':', marker='D', markersize=4)

# Add vertical line separating history and forecast
plt.axvline(x=last_date, color='red', linestyle='--', linewidth=2, alpha=0.5, label='Forecast Start')

plt.title('Historical Data + 30-Day Forecast Comparison', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontweight='bold')
plt.ylabel('Quantity (Units)', fontweight='bold')
plt.legend(fontsize=10, loc='best')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("✓ Future forecast visualization completed")

## Step 15: Save Models & Artifacts

In [ ]:
# Create models directory in Google Drive
models_dir = os.path.join('/content/drive/MyDrive/Multibliz POS System', 'trained_models')
os.makedirs(models_dir, exist_ok=True)

print("💾 Saving Models & Artifacts...\n")

# Save models
joblib.dump(prophet_model, os.path.join(models_dir, 'prophet_model.pkl'))
print(f"✓ Saved: prophet_model.pkl")

joblib.dump(xgb_model, os.path.join(models_dir, 'xgboost_model.pkl'))
print(f"✓ Saved: xgboost_model.pkl")

joblib.dump(feature_columns, os.path.join(models_dir, 'feature_columns.pkl'))
print(f"✓ Saved: feature_columns.pkl")

# Save metadata
metadata = {
    'training_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'data_points': len(daily_sales),
    'date_range': f"{daily_sales['Date'].min()} to {daily_sales['Date'].max()}",
    'prophet_mae': float(prophet_mae),
    'prophet_rmse': float(prophet_rmse),
    'xgboost_mae_test': float(xgb_mae_test),
    'xgboost_rmse_test': float(xgb_rmse_test),
    'feature_columns': feature_columns,
    'avg_daily_consumption': float(daily_sales['Quantity_Sold'].mean()),
    'max_daily_consumption': float(daily_sales['Quantity_Sold'].max())
}

joblib.dump(metadata, os.path.join(models_dir, 'model_metadata.pkl'))
print(f"✓ Saved: model_metadata.pkl")

# Save forecasts as CSV
forecast_summary.to_csv(os.path.join(models_dir, 'future_forecast_30days.csv'), index=False)
print(f"✓ Saved: future_forecast_30days.csv")

print(f"\n📁 All models saved to: {models_dir}")
print(f"\n✓ Training Complete!")

## Step 16: Generate Training Report

In [ ]:
# Generate comprehensive report
report = f"""
{"="*80}
MULTIBIZ FORECASTING MODEL TRAINING REPORT
{"="*80}

Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
Platform: Google Colab

{'='*80}
DATA SUMMARY
{'='*80}
Total Records: {len(df):,}
Total Days: {len(daily_sales)}
Date Range: {daily_sales['Date'].min()} to {daily_sales['Date'].max()}
Average Daily Consumption: {daily_sales['Quantity_Sold'].mean():.2f} units
Max Daily Consumption: {daily_sales['Quantity_Sold'].max():.0f} units

{'='*80}
MODEL PERFORMANCE
{'='*80}

PROPHET MODEL:
  MAE:  {prophet_mae:.2f} units
  RMSE: {prophet_rmse:.2f} units
  → On average, predictions deviate by ±{prophet_mae:.2f} units

XGBOOST MODEL:
  Training MAE:  {xgb_mae_train:.2f} units
  Training RMSE: {xgb_rmse_train:.2f} units
  Test MAE:      {xgb_mae_test:.2f} units
  Test RMSE:     {xgb_rmse_test:.2f} units
  → On average, test predictions deviate by ±{xgb_mae_test:.2f} units

{'='*80}
TOP 5 IMPORTANT FEATURES (XGBoost)
{'='*80}
{feature_importance.head(5).to_string(index=False)}

{'='*80}
FUTURE FORECAST (Next 30 Days)
{'='*80}
XGBoost Average:  {np.mean(xgb_future_preds):.0f} units/day
Prophet Average:  {np.mean(prophet_future_preds):.0f} units/day
Ensemble Average: {forecast_summary['Average'].mean():.0f} units/day

{'='*80}
FILES GENERATED
{'='*80}
✓ prophet_model.pkl              - Prophet trained model
✓ xgboost_model.pkl              - XGBoost trained model  
✓ feature_columns.pkl            - Feature list for predictions
✓ model_metadata.pkl             - Training metadata & metrics
✓ future_forecast_30days.csv     - 30-day forecast predictions

{'='*80}
HOW TO USE THESE MODELS
{'='*80}

1. Load Models:
   import joblib
   prophet_model = joblib.load('prophet_model.pkl')
   xgboost_model = joblib.load('xgboost_model.pkl')
   feature_columns = joblib.load('feature_columns.pkl')
   metadata = joblib.load('model_metadata.pkl')

2. Make Predictions with Prophet:
   future = prophet_model.make_future_dataframe(periods=30)
   forecast = prophet_model.predict(future)
   print(forecast[['ds', 'yhat']])

3. Make Predictions with XGBoost:
   # Prepare features as DataFrame
   features = prepare_features(data)  # Your feature preparation function
   predictions = xgboost_model.predict(features)

4. Integrate into Django:
   - Copy models to: data/models/ directory
   - Load in forecasting/management/commands/auto_generate_forecast.py
   - Run: python manage.py auto_generate_forecast

{'='*80}
THESIS DOCUMENTATION
{'='*80}

Use the following for your thesis:

METHODOLOGY CHAPTER:
- Explain data aggregation to daily consumption
- Describe feature engineering (temporal, lags, rolling statistics)
- Discuss Prophet architecture (trend + seasonality)
- Explain XGBoost gradient boosting approach

RESULTS CHAPTER:
- MAE: {xgb_mae_test:.2f} units, RMSE: {xgb_rmse_test:.2f} units
- Feature importance shows which factors drive consumption
- Predictions vs actual plots demonstrate model accuracy

DISCUSSION CHAPTER:
- Ensemble approach provides robust forecasting
- Prophet captures seasonal patterns
- XGBoost learns non-linear relationships
- Combined models improve prediction reliability

LIMITATIONS:
- Models trained on {len(df):,} historical transactions
- Accuracy depends on data quality and completeness
- External factors (events, holidays) not captured
- Requires retraining periodically with new data

FUTURE WORK:
- Incorporate external variables (holidays, marketing events)
- Implement ensemble methods with multiple models
- Add anomaly detection for demand spikes
- Develop inventory optimization based on forecasts

{'='*80}
REFERENCES
{'='*80}

Prophet:
Taylor, S. J., & Letham, B. (2018). Forecasting at scale.
The American Statistician, 72(1), 37-45.

XGBoost:
Chen, T., & Guestrin, C. (2016). XGBoost: A scalable tree boosting system.
Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge
Discovery and Data Mining.

{'='*80}
TRAINING COMPLETE
{'='*80}
"""

print(report)

# Save report
report_path = os.path.join(models_dir, 'training_report.txt')
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report)

print(f"\n✓ Report saved to: {report_path}")

## Step 17: Summary & Next Steps

In [ ]:
print("\n" + "="*80)
print("🎉 TRAINING COMPLETE!")
print("="*80)

print(f"""
✅ MODELS TRAINED:
   ✓ Prophet (Time-series with seasonality)
   ✓ XGBoost (Gradient boosting with features)
   ✓ Ensemble (Combines both for robustness)

📊 PERFORMANCE METRICS:
   Prophet MAE:    {prophet_mae:.2f} units
   XGBoost MAE:    {xgb_mae_test:.2f} units (test set)
   
📈 FORECASTS GENERATED:
   ✓ 30-day future predictions
   ✓ Confidence intervals (Prophet)
   ✓ Feature importance rankings
   
💾 FILES SAVED:
   ✓ trained_models/prophet_model.pkl
   ✓ trained_models/xgboost_model.pkl
   ✓ trained_models/feature_columns.pkl
   ✓ trained_models/model_metadata.pkl
   ✓ trained_models/future_forecast_30days.csv
   ✓ trained_models/training_report.txt

📁 LOCATION:
   Google Drive → Multibliz POS System → trained_models/

🚀 NEXT STEPS:

1. Download Models:
   - Download the trained_models folder from Google Drive
   - Place in your Django project at: data/models/

2. Integrate into Django:
   - Update forecasting/management/commands/auto_generate_forecast.py
   - Load models using joblib.load()
   - Run: python manage.py auto_generate_forecast

3. View Forecasts in Dashboard:
   - Go to: http://localhost:8000/forecasting/
   - See 30-day predictions
   - Filter by product/algorithm

4. Validate Accuracy:
   - Run validation script: python scripts/validate_forecasts.py
   - Compare predictions vs actual sales
   - Monitor MAE/RMSE metrics

5. Retraining Schedule:
   - Retrain monthly with latest data
   - Use Django management command
   - Or create a Celery task for automation

📚 DOCUMENTATION:
   - Read: trained_models/training_report.txt
   - Use outputs for thesis defense
   - Reference papers in citations section

❓ TROUBLESHOOTING:
   - Models not loading? Check file paths
   - Low accuracy? Ensure sufficient training data
   - OutOfMemory? Use smaller datasets or cloud GPU
   - Need help? Check training_report.txt for details

""")

print("="*80)
print("For questions or support, refer to the training report.")
print("="*80)